# reproject_old

Reprojects the ngc2090 attenuation `ProcessedComposite` FITS files (stored at blocked
resolution) back to the dimensions of the original HST extinction image, then
**overwrites** each composite in-place with the reprojected version.

Mirrors the blocked-header manipulation and `reprojectWrapper` logic from
`Identification/FilamentMap.py`:
- `setBlockData` scales CDELT×BF and divides CRPIX÷BF to build the blocked WCS
- `reprojectWrapper` calls `reproject_interp((data, blocked_hdr), orig_hdr, shape_out=...)`

Here we **reverse** that direction: blocked composite → original image dimensions.

**Run order**: Cell 1 (config + load) → Cell 2 (reproject all scales)

In [ ]:
# =============================================================================
# Cell 1: Imports, paths, original image
# =============================================================================
import os
import re
import warnings
import numpy as np
from pathlib import Path
from astropy.io import fits
from astropy.wcs import FITSFixedWarning
from reproject import reproject_interp

warnings.filterwarnings('ignore', category=FITSFixedWarning)

BASE_DIR   = Path(r"C:\Users\jhoffm72\Documents\FilPHANGS\Data")
GALAXY     = "ngc2090_F555W"
galaxy_dir = BASE_DIR / GALAXY
orig_path  = BASE_DIR / "OriginalImages" / "ngc2090_F555W_HST_Extinction.fits"
comp_dir   = galaxy_dir / "Composites"
out_dir    = comp_dir          # reprojected files land alongside the originals

# Load original image to get header and shape
with fits.open(orig_path) as h:
    orig_data   = np.array(h[0].data, dtype=float)
    orig_header = h[0].header.copy()

print(f"Original image shape : {orig_data.shape}")

# Determine sorted CDD scales present in the CDD folder (needed for rank -> BF)
cdd_dir    = galaxy_dir / "CDD"
cdd_scales = sorted(
    int(m.group(1))
    for f in os.listdir(cdd_dir)
    for m in [re.search(r'CDDss(\d+)pc', f)]
    if m
)
print(f"CDD scales found     : {cdd_scales}")

In [ ]:
# =============================================================================
# Cell 2: Reconstruct blocked header and reproject each ProcessedComposite
# =============================================================================

def make_blocked_header(orig_hdr, bf):
    """
    Reconstruct the WCS header for a blocked image, mirroring
    FilamentMap.setBlockData:
      CDELT *= BF   (pixel scale grows)
      CRPIX /= BF   (reference pixel shrinks)
      CD matrix diag *= BF if CD-form header
    """
    block_hdr = orig_hdr.copy()
    try:
        block_hdr['CDELT1'] = orig_hdr['CDELT1'] * bf
        block_hdr['CDELT2'] = orig_hdr['CDELT2'] * bf
    except KeyError:
        # CD-matrix form
        cd1_1 = orig_hdr['CD1_1']
        cd2_2 = orig_hdr['CD2_2']
        block_hdr['CDELT1'] = cd1_1 * bf
        block_hdr['CDELT2'] = cd2_2 * bf
        block_hdr['CD1_1']  = cd1_1 * bf
        block_hdr['CD2_2']  = cd2_2 * bf
    block_hdr['CRPIX1'] = orig_hdr['CRPIX1'] / bf
    block_hdr['CRPIX2'] = orig_hdr['CRPIX2'] / bf
    return block_hdr


def reproject_wrapper(in_data, in_header, out_header, out_shape):
    """
    Mirrors FilamentMap.reprojectWrapper:
    reproject_interp, crop NaN border, fill remaining NaN with 0.
    """
    reprojected, _ = reproject_interp(
        (in_data, in_header), out_header, shape_out=out_shape
    )
    reprojected = np.nan_to_num(reprojected, nan=0.0)
    return reprojected


for comp_file in sorted(comp_dir.glob("*_ProcessedComposite.fits")):
    m = re.search(r'CDDss(\d+)pc', comp_file.name)
    if not m:
        print(f"  Skipping (no scale found): {comp_file.name}")
        continue
    scale_pc = int(m.group(1))

    if scale_pc not in cdd_scales:
        print(f"  Skipping {scale_pc}pc: not in CDD scale list")
        continue

    rank = cdd_scales.index(scale_pc)
    bf   = 2 ** rank
    if bf <= 1:
        bf = 0   # matches FilamentMap logic (no blocking)

    with fits.open(comp_file) as h:
        comp_data = np.array(h[0].data, dtype=float)

    print(f"  {comp_file.name}")
    print(f"    scale={scale_pc}pc  rank={rank}  BF={bf}  blocked shape={comp_data.shape}")

    if bf == 0:
        # No blocking was applied; data is already at original resolution
        reprojected = comp_data
        out_hdr     = orig_header
    else:
        block_hdr   = make_blocked_header(orig_header, bf)
        reprojected = reproject_wrapper(comp_data, block_hdr,
                                         orig_header, orig_data.shape)

    fits.PrimaryHDU(
        data=reprojected.astype(np.float32),
        header=orig_header
    ).writeto(comp_file, overwrite=True)
    print(f"    -> overwritten in-place: {comp_file.name}  shape={reprojected.shape}")

print("Done.")